# Phase 2 — Step 1: Understand the NER data

**The task (Legal NER):** for every document, find the important names/things and their type, and return
`[start_char, end_char, label]`. Scoring is **strict**: start, end _and_ label must all be exactly right.

This notebook only _looks_ at the data — no model yet. We answer:

1. What exactly is in the files? (format, number of documents, labels)
2. Are the character offsets reliable? (`text[start:end]` must equal the annotated entity text)
3. How often does each label occur? (class imbalance)
4. How long are documents — in characters and in **bpe32k tokens**? (decides how we cut documents for the model, whose context is 1536 tokens)
5. Do entities overlap? Do they start/end in the middle of a token?
6. Does the text contain characters our Phase-1 cleaning would change? (if yes, we must **not** clean Phase-2 text — offsets would break)

Everything is fast (70 documents), so no caching is needed here.


## 0. Setup


In [1]:
import json, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
from tokenizers import Tokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

P2 = PROJECT_ROOT / "data" / "raw" / "Phase_2_LNER"
tokenizer = Tokenizer.from_file(str(PROJECT_ROOT / "data" / "tokenizer" / "bpe32k.json"))

def load(name):
    with open(P2 / name, encoding="utf-8") as f:
        return json.load(f)

texts = {"fold1": load("Train/fold1-text.json"), "fold2": load("Train/fold2-text.json")}
golds = {"fold1": load("Train/fold1-gold.json"), "fold2": load("Train/fold2-gold.json")}
test_texts = load("Test/fold3-text.json")
LABELS = [l.strip() for l in open(P2 / "Train" / "labels.txt", encoding="utf-8") if l.strip()]

print("labels.txt:", len(LABELS), LABELS)
for f in texts:
    print(f"{f}: {len(texts[f])} documents, gold for {len(golds[f])} documents, same ids: {set(texts[f]) == set(golds[f])}")
print(f"test (fold3): {len(test_texts)} documents")

labels.txt: 12 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
fold1: 35 documents, gold for 35 documents, same ids: True
fold2: 35 documents, gold for 35 documents, same ids: True
test (fold3): 35 documents


## 1. Are the offsets reliable?

For every gold entity `[start, end, label, entity_text]` we check `text[start:end] == entity_text`.
In Python slicing, `end` is **exclusive**: `"STATE OF HARYANA"` at 137..153 has length 153 − 137 = 16.
If every check passes, we can trust the offsets completely.


In [2]:
n_ok = n_bad = 0
bad_examples = []
unknown_labels = Counter()
for f in texts:
    for doc_id, ents in golds[f].items():
        text = texts[f][doc_id]
        for e in ents:
            s, t, lab = e[0], e[1], e[2]
            if lab not in LABELS:
                unknown_labels[lab] += 1
            if len(e) > 3 and text[s:t] != e[3]:
                n_bad += 1
                if len(bad_examples) < 5:
                    bad_examples.append((f, doc_id, s, t, lab, repr(e[3]), repr(text[s:t])))
            else:
                n_ok += 1
print(f"offset check: {n_ok:,} OK, {n_bad:,} mismatches")
for b in bad_examples:
    print("  MISMATCH", b)
print("labels in gold that are not in labels.txt:", dict(unknown_labels) or "none")

offset check: 8,393 OK, 0 mismatches
labels in gold that are not in labels.txt: none


## 2. How many entities per label? (class imbalance)

Macro-F1 gives **every label equal weight**. A rare label (few examples) counts as much as DATE — so rare labels matter a lot for our score.


In [3]:
counts = {f: Counter(e[2] for ents in golds[f].values() for e in ents) for f in golds}
total = counts["fold1"] + counts["fold2"]
print(f"{'label':10s} {'fold1':>6s} {'fold2':>6s} {'total':>6s}  {'share':>6s}  example")
examples = {}
for f in golds:
    for doc_id, ents in golds[f].items():
        for e in ents:
            examples.setdefault(e[2], texts[f][doc_id][e[0]:e[1]])
for lab in sorted(total, key=total.get, reverse=True):
    print(f"{lab:10s} {counts['fold1'][lab]:6d} {counts['fold2'][lab]:6d} {total[lab]:6d}  {total[lab]/sum(total.values()):6.1%}  {examples[lab]!r}")
print(f"\nTOTAL entities: {sum(total.values()):,}")

label       fold1  fold2  total   share  example
DATE          813    762   1575   18.8%  '19.8.2011'
STAT          525    735   1260   15.0%  'IPC'
PREC          624    559   1183   14.1%  'State of M.P. vs. Kashiram & Ors.'
COURT         455    558   1013   12.1%  'High Court of Punjab and Haryana'
AUTH          535    468   1003   12.0%  'Maharashtra Revenue Tribunal'
CASENO        423    337    760    9.1%  '(1988) 4 SCC 551'
APP           207    200    407    4.8%  'Jage Ram'
RESP          191    190    381    4.5%  'STATE OF HARYANA'
JUDGE         123    122    245    2.9%  'V. Gopala Gowda'
A.COUNSEL      70    125    195    2.3%  'Vibha Datta Makhija'
WIT            67    120    187    2.2%  'Jagdish'
R.COUNSEL      87     97    184    2.2%  'Ajay Bansal'

TOTAL entities: 8,393


## 3. How long are documents?

Our Phase-1 models read at most **1536 tokens** at once. Longer documents must be cut into **windows** (overlapping pieces).
We count characters and bpe32k tokens per document.


In [4]:
def stats(name, values):
    v = np.array(values)
    print(f"{name:28s} min {v.min():>7,}  median {int(np.median(v)):>7,}  mean {int(v.mean()):>7,}  max {v.max():>7,}")

all_docs = {**{f"{f}:{k}": t for f in texts for k, t in texts[f].items()}}
char_lens = [len(t) for t in all_docs.values()]
tok_lens = [len(tokenizer.encode(t).ids) for t in all_docs.values()]
stats("characters per train doc", char_lens)
stats("bpe32k tokens per train doc", tok_lens)
print(f"train docs longer than 1536 tokens: {sum(n > 1536 for n in tok_lens)} of {len(tok_lens)}")
test_tok = [len(tokenizer.encode(t).ids) for t in test_texts.values()]
stats("bpe32k tokens per TEST doc", test_tok)
print("empty test documents:", [k for k, t in test_texts.items() if not t.strip()])

characters per train doc     min  10,443  median  26,620  mean  31,623  max 114,044
bpe32k tokens per train doc  min   2,133  median   5,625  mean   6,648  max  23,732
train docs longer than 1536 tokens: 70 of 70
bpe32k tokens per TEST doc   min       0  median   5,990  mean   7,159  max  21,070
empty test documents: ['1305957']


## 4. Overlaps, entity length, and entities that cut a token

- **Overlap:** if two gold entities overlap, a simple "one tag per token" scheme (BIO) can't represent both.
- **Token cut:** our tokenizer splits text into pieces with exact character positions (`offsets`).
  If an entity starts or ends **inside** a token (e.g. gold "Parliament" but the token is "Parliamentary"), a token-level model can never hit the exact boundary.
  Counting these tells us the best score a token-tagging model could reach.


In [5]:
n_overlap = 0
n_cut = 0
n_total = 0
ent_tokens = []
for f in texts:
    for doc_id, ents in golds[f].items():
        text = texts[f][doc_id]
        ents = sorted(ents, key=lambda e: (e[0], e[1]))
        for a, b in zip(ents, ents[1:]):
            if b[0] < a[1]:
                n_overlap += 1
        enc = tokenizer.encode(text)
        # character positions where tokens start/end, ignoring the leading space a token may carry ("Ġword")
        starts, ends = set(), set()
        for (s, t) in enc.offsets:
            while s < t and text[s].isspace():
                s += 1
            starts.add(s); ends.add(t)
        for s, t, *_ in ents:
            n_total += 1
            if s not in starts or t not in ends:
                n_cut += 1
            ent_tokens.append(len(tokenizer.encode(text[s:t]).ids))
print(f"overlapping entity pairs: {n_overlap}")
print(f"entities whose start or end falls inside a token: {n_cut} of {n_total} ({n_cut/n_total:.2%})")
stats("tokens per entity", ent_tokens)

overlapping entity pairs: 0
entities whose start or end falls inside a token: 152 of 8393 (1.81%)
tokens per entity            min       1  median       4  mean       5  max      51


## 5. Should we clean the text like in Phase 1?

Phase-1 cleaning changed characters (curly quotes → straight, removed soft hyphens, collapsed spaces).
**Any change in Phase 2 would shift every character position after it** — and the scorer needs the original positions.
So we check what kinds of characters appear. Our byte-level tokenizer can read _any_ character, so we can simply tokenize the raw text.


In [6]:
all_text = "".join(all_docs.values())
non_ascii = Counter(ch for ch in all_text if ord(ch) > 127)
print(f"non-ASCII characters: {sum(non_ascii.values()):,} ({sum(non_ascii.values())/len(all_text):.3%})")
print("most common:", [(repr(c), n) for c, n in non_ascii.most_common(10)])
print(f"line breaks: {all_text.count(chr(10)):,} | double spaces: {all_text.count('  '):,}")

non-ASCII characters: 2,865 (0.129%)
most common: [("'…'", 590), ("'“'", 406), ("'”'", 398), ("'’'", 276), ("'\\xad'", 271), ("'‘'", 102), ("'ा'", 65), ("'–'", 55), ("'्'", 51), ("'र'", 50)]
line breaks: 0 | double spaces: 0


## 6. One document, highlighted

Gold entities are shown as `[[text|LABEL]]`, so you can see what the annotators marked — and what they did _not_ mark.


In [7]:
def highlight(text, ents, limit=1500):
    out, pos = [], 0
    for s, t, lab, *_ in sorted(ents, key=lambda e: e[0]):
        if s >= limit:
            break
        out.append(text[pos:s]); out.append(f"[[{text[s:t]}|{lab}]]"); pos = t
    out.append(text[pos:limit])
    return "".join(out)

doc_id = list(texts["fold1"])[0]
print(highlight(texts["fold1"][doc_id], golds["fold1"][doc_id]))

REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus [[STATE OF HARYANA|RESP]] ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated [[19.8.2011|DATE]] passed by the [[High Court of Punjab and Haryana|COURT]] in Criminal Appeal No.181 SB of 2000, whereby the [[High Court|COURT]] partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. [[18.11.1994|DATE]], at about 8.00 A.M. in the morning the complainant [[Jagdish|WIT]] (PW-5) along with his two sons namely Sukhbir and [[Mange Ram|WIT]] (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, [[Jage Ram|APP]] (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, ent

## What to paste back

The outputs of cells 1–5 (offset check, label table, length stats, overlap/cut numbers, non-ASCII check).
